In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random

# Setting seed for reproducibility.
np.random.seed(42)
random.seed(42)

print("Libraries loaded successfully.")
print(f"Generation date: {datetime.now().strftime('%Y-%m-%d')}")

Libraries loaded successfully.
Generation date: 2026-06-12


In [2]:
# Reference Data for Tokai Logistics Operational Parameters

# Five origin hubs across Greater Tokyo area.
hubs = [
    'Shibuya Hub',
    'Shinagawa Hub', 
    'Yokohama Hub',
    'Chiba Hub',
    'Kawasaki Hub'
]

# Three prefectures with realistic destination wards.
# Wards are administrative districts in each prefecture.
destinations = {
    'Tokyo': [
        'Shinjuku', 'Shibuya', 'Minato', 'Chiyoda',
        'Setagaya', 'Nerima', 'Adachi', 'Koto'
    ],
    'Kanagawa': [
        'Yokohama-Nishi', 'Yokohama-Tsurumi',
        'Kawasaki-Nakahara', 'Sagamihara'
    ],
    'Chiba': [
        'Chiba-Chuo', 'Funabashi',
        'Ichikawa', 'Matsudo'
    ]
}

# Delivery time windows created with standard Japanese logistics industry format
# to mimic Yamato Holdings' actual delivery window structure.
delivery_windows = {
    'Morning':       ('08:00', '12:00'),
    'Afternoon':     ('12:00', '14:00'),
    'Late Afternoon':('14:00', '16:00'),
    'Evening':       ('16:00', '18:00'),
    'Night':         ('18:00', '21:00')
}

# Creating weather conditions with realistic frequency weights for Tokyo.
# Tokyo averages roughly 1,500mm of rain annually.
# Snow is rare but impactful.
weather_conditions = ['Clear', 'Cloudy', 'Rain', 'Heavy Rain', 'Snow']
weather_weights =    [0.45,    0.30,     0.15,   0.07,         0.03]

# Creating delivery vehicle types by hub to reflect realistic urban logistics mix.
# Motorcycles and bicycles for dense urban areas and vans for heavier packages/suburban routes.
vehicle_types = ['Van', 'Motorcycle', 'Bicycle']

# Hub to vehicle type distribution.
# Shibuya and Shinjuku are dense urban areas, more motorcycles and bicycles.
# Yokohama and Chiba are more suburban, more vans.
hub_vehicle_weights = {
    'Shibuya Hub':   [0.30, 0.45, 0.25],
    'Shinagawa Hub': [0.40, 0.40, 0.20],
    'Yokohama Hub':  [0.60, 0.30, 0.10],
    'Chiba Hub':     [0.65, 0.25, 0.10],
    'Kawasaki Hub':  [0.50, 0.35, 0.15]
}

# Hub to prefecture routing to determine which hubs serve which prefectures
# to reflect realistic geographic proximity between locations.
hub_prefecture_weights = {
    'Shibuya Hub':   {'Tokyo': 0.80, 'Kanagawa': 0.15, 'Chiba': 0.05},
    'Shinagawa Hub': {'Tokyo': 0.60, 'Kanagawa': 0.35, 'Chiba': 0.05},
    'Yokohama Hub':  {'Tokyo': 0.20, 'Kanagawa': 0.75, 'Chiba': 0.05},
    'Chiba Hub':     {'Tokyo': 0.25, 'Kanagawa': 0.05, 'Chiba': 0.70},
    'Kawasaki Hub':  {'Tokyo': 0.40, 'Kanagawa': 0.55, 'Chiba': 0.05}
}

print("Reference data defined successfully.")
print(f"\nHubs: {len(hubs)}")
print(f"Prefectures: {len(destinations)}")
print(f"Total destination wards: {sum(len(v) for v in destinations.values())}")
print(f"Delivery windows: {len(delivery_windows)}")
print(f"Weather conditions: {len(weather_conditions)}")

Reference data defined successfully.

Hubs: 5
Prefectures: 3
Total destination wards: 16
Delivery windows: 5
Weather conditions: 5


In [3]:
# Main dataset generation: 50,000 delivery records

# Delay parameters by condition.
# Base delay minutes follow a realistic right-skewed distribution.
# Most deliveries are on time or slightly late.
# A small percentage have significant delays.

delay_params = {
    # (mean_delay, std_dev, on_time_probability)
    # On-time means delay <= 0 minutes
    'Clear':      (5,  15, 0.82),
    'Cloudy':     (8,  18, 0.78),
    'Rain':       (18, 25, 0.65),
    'Heavy Rain': (35, 30, 0.45),
    'Snow':       (55, 40, 0.30)
}

# Window-based delay multipliers.
# Morning window often most reliable.
# Evening and night have higher delay risk due to traffic.
window_delay_multipliers = {
    'Morning':        0.85,
    'Afternoon':      1.00,
    'Late Afternoon': 1.10,
    'Evening':        1.25,
    'Night':          1.15
}

# Creating monthly volume weights to reflect Japanese logistics seasonality.
# Peak periods: March (fiscal year end), July-August (summer gifts)
# December (Christmas/New Year), January (New Year returns).
monthly_weights = {
    1: 0.095,  # January - New Year gift returns
    2: 0.075,  # February - quiet period
    3: 0.090,  # March - fiscal year end
    4: 0.075,  # April - stable
    5: 0.080,  # May - Golden Week
    6: 0.075,  # June - stable
    7: 0.090,  # July - Ochugen gift season
    8: 0.085,  # August - summer peak
    9: 0.075,  # September - stable
    10: 0.080, # October - stable
    11: 0.085, # November - pre-holiday
    12: 0.095  # December - Christmas/Oseibo peak
}

# Creating 12% realistic failure rate.
# Based on Japan MLIT first-attempt delivery failure statistics.

records = []
package_id = 1

# Revised delay parameters for more realistic failure rates.
# On-time probability reduced to reflect real-world conditions.
delay_params = {
    # (mean_delay, std_dev, on_time_probability, failure_probability)
    'Clear':      (5,  15, 0.72, 0.06),
    'Cloudy':     (8,  18, 0.68, 0.08),
    'Rain':       (18, 25, 0.55, 0.14),
    'Heavy Rain': (35, 30, 0.38, 0.22),
    'Snow':       (55, 40, 0.22, 0.35)
}

# Adding window failure probability modifiers.
# Evening and night windows have higher failure rates
# as recipients are more likely to be unavailable or absent.
window_failure_modifiers = {
    'Morning':        0.85,
    'Afternoon':      0.90,
    'Late Afternoon': 1.00,
    'Evening':        1.15,
    'Night':          1.25
}

for month, weight in monthly_weights.items():
    month_count = int(50000 * weight)
    variation = random.uniform(-0.06, 0.06)
    month_count = int(month_count * (1 + variation))

    for _ in range(month_count):
        hub = random.choice(hubs)
        prefecture = random.choices(
            list(hub_prefecture_weights[hub].keys()),
            weights=list(hub_prefecture_weights[hub].values())
        )[0]
        ward = random.choice(destinations[prefecture])
        window = random.choice(list(delivery_windows.keys()))
        window_start, window_end = delivery_windows[window]

        if month in [4, 6, 9, 11]:
            max_day = 30
        elif month == 2:
            max_day = 29
        else:
            max_day = 31
        day = random.randint(1, max_day)

        start_hour = int(window_start.split(':')[0])
        end_hour = int(window_end.split(':')[0])
        scheduled_hour = random.randint(start_hour, end_hour - 1)
        scheduled_minute = random.randint(0, 59)
        scheduled_dt = datetime(2024, month, day,
                               scheduled_hour, scheduled_minute)

        weather = random.choices(weather_conditions,
                                weights=weather_weights)[0]

        base_mean, base_std, on_time_prob, fail_prob = \
            delay_params[weather]
        window_mult = window_delay_multipliers[window]
        window_fail_mod = window_failure_modifiers[window]

        # Determine delivery outcome,
        outcome_roll = random.random()
        adjusted_fail_prob = fail_prob * window_fail_mod

        if outcome_roll < on_time_prob:
            status = 'On Time'
            delay_minutes = int(random.uniform(-15, 5))
        elif outcome_roll < on_time_prob + adjusted_fail_prob:
            status = 'Failed'
            delay_minutes = int(abs(np.random.normal(
                base_mean * window_mult * 1.5, base_std)))
        else:
            # Delayed but delivered.
            delay_minutes = int(abs(np.random.normal(
                base_mean * window_mult, base_std)))
            if delay_minutes <= 30:
                status = 'Minor Delay'
            elif delay_minutes <= 60:
                status = 'Moderate Delay'
            else:
                status = 'Major Delay'

        actual_dt = scheduled_dt + timedelta(minutes=delay_minutes)

        # Attempt number.
        if status == 'Failed':
            attempt = random.choices(
                [1, 2, 3], weights=[0.60, 0.30, 0.10])[0]
        else:
            attempt = 1

        vehicle = random.choices(
            vehicle_types,
            weights=hub_vehicle_weights[hub]
        )[0]

        weight_kg = round(random.choices(
            [random.uniform(0.1, 1.0),
             random.uniform(1.0, 5.0),
             random.uniform(5.0, 20.0)],
            weights=[0.50, 0.35, 0.15]
        )[0], 2)

        redelivery_cost = 800 if status == 'Failed' else 0
        sla_penalty = 500 if delay_minutes > 60 else 0

        driver_id = f"{hub[:3].upper()}-{random.randint(1, 50):03d}"

        records.append({
            'Package_ID': f'TKL-2024-{package_id:06d}',
            'Origin_Hub': hub,
            'Destination_Prefecture': prefecture,
            'Destination_Ward': ward,
            'Delivery_Window': window,
            'Scheduled_Delivery_DateTime': scheduled_dt,
            'Actual_Delivery_DateTime': actual_dt,
            'Delay_Minutes': delay_minutes,
            'Delivery_Status': status,
            'Weather_Condition': weather,
            'Vehicle_Type': vehicle,
            'Package_Weight_kg': weight_kg,
            'Delivery_Attempt_Number': attempt,
            'Driver_ID': driver_id,
            'Redelivery_Cost_JPY': redelivery_cost,
            'SLA_Penalty_JPY': sla_penalty,
            'Total_Revenue_Impact_JPY': redelivery_cost + sla_penalty
        })
        package_id += 1

df = pd.DataFrame(records)

# Calculating actual failure rate.
failure_rate = len(df[df['Delivery_Status'] == 'Failed']) / len(df)

print(f"Dataset shape: {df.shape}")
print(f"\nDelivery status distribution:")
print(df['Delivery_Status'].value_counts())
print(f"\nActual failure rate: {failure_rate:.1%}")
print(f"\nWeather distribution:")
print(df['Weather_Condition'].value_counts())
print(f"\nTotal revenue impact: ¥{df['Total_Revenue_Impact_JPY'].sum():,.0f}")
print(f"  Redelivery costs: ¥{df['Redelivery_Cost_JPY'].sum():,.0f}")
print(f"  SLA penalties: ¥{df['SLA_Penalty_JPY'].sum():,.0f}")
print(f"\nMonthly distribution:")
df['Month'] = pd.to_datetime(
    df['Scheduled_Delivery_DateTime']).dt.month
print(df.groupby('Month').size())

Dataset shape: (50279, 17)

Delivery status distribution:
Delivery_Status
On Time           32570
Minor Delay        9866
Failed             5023
Moderate Delay     2079
Major Delay         741
Name: count, dtype: int64

Actual failure rate: 10.0%

Weather distribution:
Weather_Condition
Clear         22862
Cloudy        14862
Rain           7432
Heavy Rain     3556
Snow           1567
Name: count, dtype: int64

Total revenue impact: ¥4,825,900
  Redelivery costs: ¥4,018,400
  SLA penalties: ¥807,500

Monthly distribution:
Month
1     4829
2     3836
3     4360
4     3770
5     4093
6     3529
7     4307
8     4480
9     3914
10    4108
11    4403
12    4650
dtype: int64


In [4]:
import os
os.makedirs("data/raw", exist_ok=True)

df.to_csv("data/raw/tokai_logistics_deliveries.csv", index=False)

print("Dataset saved successfully.")
print(f"\nFile: data/raw/tokai_logistics_deliveries.csv")
print(f"Total records: {len(df):,}")
print(f"Total columns: {len(df.columns)}")
print(f"\nColumns saved:")
for col in df.columns:
    print(f"  {col}")
print(f"\nDate range: {df['Scheduled_Delivery_DateTime'].min()} "
      f"to {df['Scheduled_Delivery_DateTime'].max()}")
print(f"\nRevenue impact summary:")
print(f"  Failed deliveries: "
      f"{len(df[df['Delivery_Status'] == 'Failed']):,}")
print(f"  Total redelivery cost: "
      f"¥{df['Redelivery_Cost_JPY'].sum():,.0f}")
print(f"  Total SLA penalties: "
      f"¥{df['SLA_Penalty_JPY'].sum():,.0f}")
print(f"  Combined revenue impact: "
      f"¥{df['Total_Revenue_Impact_JPY'].sum():,.0f}")

Dataset saved successfully.

File: data/raw/tokai_logistics_deliveries.csv
Total records: 50,279
Total columns: 18

Columns saved:
  Package_ID
  Origin_Hub
  Destination_Prefecture
  Destination_Ward
  Delivery_Window
  Scheduled_Delivery_DateTime
  Actual_Delivery_DateTime
  Delay_Minutes
  Delivery_Status
  Weather_Condition
  Vehicle_Type
  Package_Weight_kg
  Delivery_Attempt_Number
  Driver_ID
  Redelivery_Cost_JPY
  SLA_Penalty_JPY
  Total_Revenue_Impact_JPY
  Month

Date range: 2024-01-01 08:13:00 to 2024-12-31 20:59:00

Revenue impact summary:
  Failed deliveries: 5,023
  Total redelivery cost: ¥4,018,400
  Total SLA penalties: ¥807,500
  Combined revenue impact: ¥4,825,900


In [5]:
# AWS S3 Upload of Tokai Logistics Dataset.
# Using boto3 to upload the cleaned dataset to S3
# to demonstrate cloud pipeline integration step.

import boto3
from botocore.exceptions import ClientError, NoCredentialsError

# S3 configuration
BUCKET_NAME = 'tokyo-last-mile-delivery-performance-analysis-bucket'
LOCAL_FILE_PATH = 'data/raw/tokai_logistics_deliveries.csv'
S3_KEY = 'raw/tokai_logistics_deliveries.csv'

# Initializing S3 client, boto3 automatically uses credentials from aws configurement.
s3_client = boto3.client('s3', region_name='us-east-2')

print("S3 client initialized.")
print(f"Target bucket: {BUCKET_NAME}")
print(f"Local file: {LOCAL_FILE_PATH}")
print(f"S3 destination key: {S3_KEY}")

S3 client initialized.
Target bucket: tokyo-last-mile-delivery-performance-analysis-bucket
Local file: data/raw/tokai_logistics_deliveries.csv
S3 destination key: raw/tokai_logistics_deliveries.csv


In [7]:
# Uploading dataset to S3 with error handling.
import os

try:
    # Verifying local file exists before attempting upload.
    if not os.path.exists(LOCAL_FILE_PATH):
        raise FileNotFoundError(
            f"Local file not found: {LOCAL_FILE_PATH}")
    
    file_size = os.path.getsize(LOCAL_FILE_PATH) / (1024 * 1024)
    print(f"Local file confirmed: {file_size:.2f} MB")
    print(f"Beginning upload to s3://{BUCKET_NAME}/{S3_KEY}...")
    
    # Performing upload.
    s3_client.upload_file(
        Filename=LOCAL_FILE_PATH,
        Bucket=BUCKET_NAME,
        Key=S3_KEY
    )
    
    print(f"\n✓ Upload successful")
    
    # Verifying upload by retrieving file metadata.
    response = s3_client.head_object(
        Bucket=BUCKET_NAME, Key=S3_KEY)
    
    print(f"\nS3 object verification:")
    print(f"  Size: {response['ContentLength']:,} bytes")
    print(f"  Last modified: {response['LastModified']}")
    print(f"  Content type: {response.get('ContentType', 'N/A')}")
    print(f"  ETag: {response['ETag']}")

except FileNotFoundError as e:
    print(f"✗ Error: {e}")
except NoCredentialsError:
    print("✗ Error: AWS credentials not found.")
    print("  Run 'aws configure' in Command Prompt to set them up.")
except ClientError as e:
    error_code = e.response['Error']['Code']
    error_message = e.response['Error']['Message']
    print(f"✗ AWS Client Error [{error_code}]: {error_message}")
except Exception as e:
    print(f"✗ Unexpected error: {type(e).__name__}: {e}")

Local file confirmed: 6.99 MB
Beginning upload to s3://tokyo-last-mile-delivery-performance-analysis-bucket/raw/tokai_logistics_deliveries.csv...

✓ Upload successful

S3 object verification:
  Size: 7,331,715 bytes
  Last modified: 2026-06-12 17:35:20+00:00
  Content type: binary/octet-stream
  ETag: "2f21cf5e9eb995d7158e2030a46e627a"
